# Pokemon PokeAPI CSV Project

This project builds a complete local analytics pipeline using Pokémon data from the PokeAPI CSV dataset. The workflow reads raw CSV files stored on a local machine, performs data cleaning and validation using Python and pandas, loads the cleaned tables into a DuckDB database, and executes SQL-based analytical queries for exploration and reporting.

The project demonstrates a full end-to-end data engineering and analytics workflow, including:

- Local file ingestion
- Data cleaning and transformation
- Relational database modeling
- Data quality validation
- SQL view creation
- Analytical querying
- Exporting results for downstream analysis

### Import Libraries
This cell imports the Python libraries needed for the project. `os` is used to work with local file paths, `pandas` is used to read and clean CSV data, `duckdb` is used to create and query a local database, and warnings is used to suppress unnecessary warning messages.

In [ ]:
# Import libraries for data handling
import os
import pandas as pd
import duckdb

### Set file paths
This cell defines where the raw CSV files are stored on the computer and names the DuckDB database file that will be created. The `data_dir` variable points to the local folder containing the Pokémon CSV files, while `duckdb_file` stores the name of the output database.

In [2]:
# This is the folder where all of the raw Pokémon CSV files are stored
data_dir = os.path.expanduser("/Users/jaykim/Documents/MEDS/EDS-2026-Q2/EDS-213/lab/eds213-database-lab/data")

#This is the name of the DuckDB database file that will be created
duckdb_file = "pokemon_project.duckdb"

### Create Helper Functions for Reading CSV Files

This section defines reusable helper functions for working with CSV files.

The `csv_path()` function dynamically creates the full file path for a CSV file using the base data directory.

The `read_poke_csv()` function:
- Builds the file path,
- Validates that the file exists,
- Reads the CSV into a pandas DataFrame,
- Standardizes missing values such as `"NA"` and `"NULL"`.

These helper functions reduce repetitive code and make the data ingestion process more consistent.

In [ ]:
def csv_path(file_name):
    """
    Build the full path to a CSV file.

    Parameters
    ----------
    file_name : str
        The name of the CSV file without the .csv extension.

    Returns
    -------
    str
        The full file path to the CSV file.
    """

    # Combine the data directory, file name, and .csv extension
    return os.path.join(data_dir, f"{file_name}.csv")


def read_poke_csv(file_name):
    """
    Read a Pokémon CSV file into a pandas DataFrame.

    This function does three things:
    1. Builds the full file path.
    2. Checks whether the file exists.
    3. Reads the CSV into pandas.

    Parameters
    ----------
    file_name : str
        The name of the CSV file without the .csv extension.

    Returns
    -------
    pandas.DataFrame
        The loaded CSV file.
    """

    # Create the full path to the CSV file
    path = csv_path(file_name)

    # Stop the code if the file does not exist
    # This prevents confusing errors later in the workflow
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    # Print which file is being read so we can track progress
    print(f"Reading: {path}")

    # Read the CSV file.
    # The na_values argument tells pandas to treat "", "NA", and "NULL" as missing values
    return pd.read_csv(path, na_values=["", "NA", "NULL"])

### Create Data-Cleaning and Validation Helper Functions

This section defines utility functions used throughout the data-cleaning workflow.

The helper functions perform three major tasks:

1. Convert inconsistent boolean-like values into standardized Python logical values (`True` and `False`).

2. Validate primary keys by ensuring:
   - no missing values exist,
   - no duplicate IDs exist.

3. Validate foreign key relationships by checking whether child table values exist in their related parent tables.

These functions help improve overall data quality and database integrity before loading the data into DuckDB.

In [ ]:
def to_logical(x):
    """
    Convert different boolean-like formats into True, False, or None.

    The raw data may store logical values in different ways:
    - 1 or "true" should become True
    - 0 or "false" should become False
    - unclear values should become None
    """

    # If the value is already a Python boolean, return it directly
    if isinstance(x, bool):
        return x

    # If the value is numeric, treat 1 as True and everything else as False
    if isinstance(x, (int, float)):
        return x == 1

    # Convert the value to lowercase text so comparisons are consistent
    x_str = str(x).lower()

    # These values are interpreted as True
    if x_str in ("1", "true", "t", "yes"):
        return True

    # These values are interpreted as False
    elif x_str in ("0", "false", "f", "no"):
        return False

    # If the value does not match a known true/false pattern, return None
    else:
        return None


def check_primary_key(df, key_col, table_name):
    """
    Check whether a table has a valid primary key.

    A primary key should:
    1. Never be missing.
    2. Never be duplicated.
    """

    # Check for missing values in the primary key column
    if df[key_col].isna().any():
        raise ValueError(f"{table_name}: primary key contains missing values.")

    # Count how many times each primary key value appears
    dupes = df[key_col].value_counts()

    # If any key appears more than once, the primary key is invalid
    if (dupes > 1).any():
        raise ValueError(f"{table_name}: primary key contains duplicate values.")

    # Print confirmation if the primary key passes both checks
    print(f"PK check passed: {table_name}.{key_col}")


def check_foreign_key(child_df, child_col, parent_df, parent_col, child_table, parent_table):
    """
    Check whether a foreign key relationship is valid.

    A foreign key is valid when every non-missing value in the child table
    exists in the related parent table.
    """

    # Keep only non-missing child foreign key values
    child_vals = child_df[child_df[child_col].notna()][child_col]

    # Store parent key values as a set for faster lookup
    parent_vals = set(parent_df[parent_col].unique())

    # Find child values that do not appear in the parent table
    orphan_rows = child_vals[~child_vals.isin(parent_vals)]

    # If orphan values exist, print a warning
    # This does not stop the code because some datasets may contain imperfect relationships
    if len(orphan_rows) > 0:
        print(f"Warning: {child_table}.{child_col} has "
              f"{len(orphan_rows)} orphan values not found in "
              f"{parent_table}.{parent_col}")

    # If no orphan values exist, the relationship passes
    else:
        print(f"FK check passed: {child_table}.{child_col} -> "
              f"{parent_table}.{parent_col}")

### Read Raw CSV Files

This section loads all raw Pokémon CSV files into pandas DataFrames.

Each dataset is stored in a separate raw DataFrame so the original source data remains unchanged during the cleaning process.

The project imports multiple relational datasets, including:
- Pokémon species,
- Pokémon information,
- types,
- moves,
- abilities,
- stats,
- and relationship tables connecting these entities.

These raw tables serve as the starting point for the ETL pipeline.

In [ ]:
# Each CSV is read into a raw DataFrame
# These raw DataFrames preserve the original structure of the source data
raw_pokemon_species = read_poke_csv("pokemon_species")
raw_pokemon = read_poke_csv("pokemon")
raw_types = read_poke_csv("types")
raw_pokemon_types = read_poke_csv("pokemon_types")
raw_moves = read_poke_csv("moves")
raw_pokemon_moves = read_poke_csv("pokemon_moves")
raw_abilities = read_poke_csv("abilities")
raw_pokemon_abilities = read_poke_csv("pokemon_abilities")
raw_stats = read_poke_csv("stats")
raw_pokemon_stats = read_poke_csv("pokemon_stats")

Reading: /Users/jaykim/Documents/MEDS/EDS-2026-Q2/EDS-213/lab/eds213-database-lab/data/pokemon_species.csv
Reading: /Users/jaykim/Documents/MEDS/EDS-2026-Q2/EDS-213/lab/eds213-database-lab/data/pokemon.csv
Reading: /Users/jaykim/Documents/MEDS/EDS-2026-Q2/EDS-213/lab/eds213-database-lab/data/types.csv
Reading: /Users/jaykim/Documents/MEDS/EDS-2026-Q2/EDS-213/lab/eds213-database-lab/data/pokemon_types.csv
Reading: /Users/jaykim/Documents/MEDS/EDS-2026-Q2/EDS-213/lab/eds213-database-lab/data/moves.csv
Reading: /Users/jaykim/Documents/MEDS/EDS-2026-Q2/EDS-213/lab/eds213-database-lab/data/pokemon_moves.csv
Reading: /Users/jaykim/Documents/MEDS/EDS-2026-Q2/EDS-213/lab/eds213-database-lab/data/abilities.csv
Reading: /Users/jaykim/Documents/MEDS/EDS-2026-Q2/EDS-213/lab/eds213-database-lab/data/pokemon_abilities.csv
Reading: /Users/jaykim/Documents/MEDS/EDS-2026-Q2/EDS-213/lab/eds213-database-lab/data/stats.csv
Reading: /Users/jaykim/Documents/MEDS/EDS-2026-Q2/EDS-213/lab/eds213-database-lab/d

### Clean `pokemon_species`

This section cleans and standardizes the Pokémon species dataset.

- selects only relevant columns,
- renames columns for clarity and consistency,
- converts identifier columns into nullable integer types,
- converts legendary and mythical indicators into logical boolean values.

The resulting table becomes the master reference table for Pokémon species information.

In [ ]:
# Select only the columns needed for the final database
pokemon_species = raw_pokemon_species[["id", 
                                       "identifier",
                                       "generation_id",
                                       "evolves_from_species_id",
                                       "is_legendary",
                                       "is_mythical"]].copy()

# Rename columns to make them clearer and more database-friendly
pokemon_species.columns = ["species_id",
                           "identifier",
                           "generation_id",
                           "evolves_from_species_id",
                           "is_legendary",
                           "is_mythical"]

# Convert ID columns to nullable integers
# Int64 allows missing values, unlike regular int64
pokemon_species["species_id"] = pokemon_species["species_id"].astype("Int64")
pokemon_species["generation_id"] = pokemon_species["generation_id"].astype("Int64")
pokemon_species["evolves_from_species_id"] = pokemon_species["evolves_from_species_id"].astype("Int64")

# Convert legendary and mythical fields into True/False values
pokemon_species["is_legendary"] = pokemon_species["is_legendary"].apply(to_logical)
pokemon_species["is_mythical"] = pokemon_species["is_mythical"].apply(to_logical)

### Clean `pokemon`

This section cleans the main Pokémon table.

- selects key Pokémon attributes,
- renames the primary key column to `pokemon_id`,
- standardizes numeric data types,
- converts the `is_default` field into a boolean value.

This table stores core Pokémon-level information such as:
- species relationship,
- height,
- weight,
- and base experience.

In [ ]:
# Select the columns needed for the pokemon table
pokemon = raw_pokemon[["id",
                       "species_id",
                       "identifier",
                       "height",
                       "weight",
                       "base_experience",
                       "is_default"]].copy()

# Rename id to pokemon_id so it is clear this ID belongs to the pokemon table
pokemon.columns = ["pokemon_id",
                   "species_id",
                   "identifier",
                   "height",
                   "weight",
                   "base_experience",
                   "is_default"]

# Convert numeric columns to nullable integers
pokemon["pokemon_id"] = pokemon["pokemon_id"].astype("Int64")
pokemon["species_id"] = pokemon["species_id"].astype("Int64")
pokemon["height"] = pokemon["height"].astype("Int64")
pokemon["weight"] = pokemon["weight"].astype("Int64")
pokemon["base_experience"] = pokemon["base_experience"].astype("Int64")

# Convert is_default into a logical True/False field
pokemon["is_default"] = pokemon["is_default"].apply(to_logical)

### Clean `types`

This section creates a cleaned Pokémon type reference table.

- keeps only the type ID and type name,
- renames the primary key column,
- converts the ID column into a nullable integer type.

This table is later used to connect Pokémon and moves to their elemental types.

In [ ]:
# Keep only the type ID and type name
types = raw_types[["id", "identifier"]].copy()

# Rename id to type_id for clarity
types.columns = ["type_id", "identifier"]

# Convert type_id to a nullable integer
types["type_id"] = types["type_id"].astype("Int64")

### Clean `pokemon_types`

This section cleans the relationship table connecting Pokémon to their elemental types.

Because a Pokémon may have multiple types, this table represents a many-to-many relationship.

- standardizes ID columns,
- sorts records for reproducibility,
- creates an artificial primary key,
- reorganizes columns into a database-friendly structure.

This table is later used to identify primary and secondary Pokémon types.

In [ ]:
# This table connects pokemon to their types
# A Pokémon can have more than one type, so this is a relationship table
pokemon_types = raw_pokemon_types[["pokemon_id", "type_id", "slot"]].copy()

# Convert relationship columns to nullable integers
pokemon_types["pokemon_id"] = pokemon_types["pokemon_id"].astype("Int64")
pokemon_types["type_id"] = pokemon_types["type_id"].astype("Int64")
pokemon_types["slot"] = pokemon_types["slot"].astype("Int64")

# Sort rows so the generated pokemon_type_id is consistent every time the code runs
pokemon_types = pokemon_types.sort_values(["pokemon_id", "slot", "type_id"]).reset_index(drop = True)

# Create a new artificial primary key for this relationship table
pokemon_types["pokemon_type_id"] = range(1, len(pokemon_types) + 1)

# Reorder columns so the primary key appears first
pokemon_types = pokemon_types[["pokemon_type_id", "pokemon_id", "type_id", "slot"]]

### Clean `moves`

This section cleans the Pokémon moves dataset.

- selects move-related attributes,
- renames the move identifier column,
- standardizes numeric data types.

The resulting table stores information such as:
- move power,
- accuracy,
- PP,
- damage class,
- and elemental type.

In [ ]:
# Select columns describing each Pokémon move
moves = raw_moves[["id",
                   "identifier",
                   "type_id",
                   "power",
                   "accuracy",
                   "pp",
                   "damage_class_id"]].copy()

# Rename id to move_id
moves.columns = ["move_id",
                 "identifier",
                 "type_id",
                 "power",
                 "accuracy",
                 "pp",
                 "damage_class_id"]

# Convert move attributes to nullable integers
moves["move_id"] = moves["move_id"].astype("Int64")
moves["type_id"] = moves["type_id"].astype("Int64")
moves["power"] = moves["power"].astype("Int64")
moves["accuracy"] = moves["accuracy"].astype("Int64")
moves["pp"] = moves["pp"].astype("Int64")
moves["damage_class_id"] = moves["damage_class_id"].astype("Int64")

### Clean `pokemon_moves`

This section cleans the relationship table connecting Pokémon to the moves they can learn.

- standardizes column names,
- converts numeric fields into nullable integers,
- sorts rows for reproducibility,
- creates a unique artificial primary key.

This table captures move-learning relationships across Pokémon and game versions.

In [ ]:
# This table connects Pokémon to the moves they can learn
pokemon_moves = raw_pokemon_moves[["pokemon_id",
                                   "move_id",
                                   "version_group_id",
                                   "level",
                                   "pokemon_move_method_id"]].copy()

# Rename pokemon_move_method_id to move_learn_method_id for easier reading
pokemon_moves.columns = ["pokemon_id",
                         "move_id",
                         "version_group_id",
                         "level",
                         "move_learn_method_id"]

# Convert all ID and numeric columns to nullable integers
pokemon_moves["pokemon_id"] = pokemon_moves["pokemon_id"].astype("Int64")
pokemon_moves["move_id"] = pokemon_moves["move_id"].astype("Int64")
pokemon_moves["version_group_id"] = pokemon_moves["version_group_id"].astype("Int64")
pokemon_moves["level"] = pokemon_moves["level"].astype("Int64")
pokemon_moves["move_learn_method_id"] = pokemon_moves["move_learn_method_id"].astype("Int64")

# Sort rows before assigning a new artificial primary key
pokemon_moves = pokemon_moves.sort_values(["pokemon_id",
                                           "move_id",
                                           "version_group_id",
                                           "level",
                                           "move_learn_method_id"]).reset_index(drop = True)

# Create a unique primary key for the pokemon_moves relationship table
pokemon_moves["pokemon_move_id"] = range(1, len(pokemon_moves) + 1)

# Reorder columns so the primary key is first
pokemon_moves = pokemon_moves[["pokemon_move_id",
                               "pokemon_id",
                               "move_id",
                               "version_group_id",
                               "level",
                               "move_learn_method_id"]]

### Clean `abilities`

This section creates a cleaned abilities reference table.

- keeps only ability identifiers and names,
- renames the ID column,
- converts identifiers into nullable integer format.

This table is later used to connect Pokémon to their abilities.

In [ ]:
# Keep only the ability ID and ability name
abilities = raw_abilities[["id", "identifier"]].copy()

# Rename id to ability_id
abilities.columns = ["ability_id", "identifier"]

# Convert ability_id to a nullable integer
abilities["ability_id"] = abilities["ability_id"].astype("Int64")

### Clean `pokemon_abilities`

This section cleans the relationship table connecting Pokémon to their abilities.

- standardizes ID columns,
- converts hidden ability indicators into boolean values,
- sorts records,
- creates a unique artificial primary key.

This table captures both standard and hidden Pokémon abilities.

In [ ]:
# This table connects Pokémon to their possible abilities
pokemon_abilities = raw_pokemon_abilities[["pokemon_id", "ability_id", "is_hidden", "slot"]].copy()

# Convert ID and slot columns to nullable integers
pokemon_abilities["pokemon_id"] = pokemon_abilities["pokemon_id"].astype("Int64")
pokemon_abilities["ability_id"] = pokemon_abilities["ability_id"].astype("Int64")
pokemon_abilities["slot"] = pokemon_abilities["slot"].astype("Int64")

# Convert is_hidden to True/False
pokemon_abilities["is_hidden"] = pokemon_abilities["is_hidden"].apply(to_logical)

# Sort before generating the artificial primary key
pokemon_abilities = pokemon_abilities.sort_values(["pokemon_id", "slot", "ability_id"]).reset_index(drop = True)

# Create a unique primary key for this relationship table
pokemon_abilities["pokemon_ability_id"] = range(1, len(pokemon_abilities) + 1)

# Reorder columns
pokemon_abilities = pokemon_abilities[["pokemon_ability_id", "pokemon_id", "ability_id", "is_hidden", "slot"]]

### Clean `stats`

This section creates a cleaned Pokémon stats reference table.

- keeps only stat IDs and names,
- renames the identifier column,
- converts IDs into nullable integer format.

The table stores stat categories such as:
- HP,
- attack,
- defense,
- speed,
- special attack,
- and special defense.

In [ ]:
# Keep only stat ID and stat name
stats = raw_stats[["id", "identifier"]].copy()

# Rename id to stat_id
stats.columns = ["stat_id", "identifier"]

# Convert stat_id to nullable integer
stats["stat_id"] = stats["stat_id"].astype("Int64")

### Clean `pokemon_stats`

This section cleans the table containing Pokémon base stat values.

- standardizes numeric columns,
- sorts rows,
- creates an artificial primary key,
- reorganizes columns into a relational database structure.

This table is later used to calculate total base stats and comparative Pokémon strength metrics.

In [ ]:
# This table stores the base stat values for each Pokémon
pokemon_stats = raw_pokemon_stats[["pokemon_id", "stat_id", "base_stat"]].copy()

# Convert columns to nullable integers
pokemon_stats["pokemon_id"] = pokemon_stats["pokemon_id"].astype("Int64")
pokemon_stats["stat_id"] = pokemon_stats["stat_id"].astype("Int64")
pokemon_stats["base_stat"] = pokemon_stats["base_stat"].astype("Int64")

# Sort before creating the artificial primary key
pokemon_stats = pokemon_stats.sort_values(["pokemon_id", "stat_id"]).reset_index(drop = True)

# Create a unique primary key for this table
pokemon_stats["pokemon_stat_id"] = range(1, len(pokemon_stats) + 1)

# Reorder columns
pokemon_stats = pokemon_stats[["pokemon_stat_id", "pokemon_id", "stat_id", "base_stat"]]

### Check Primary Keys

This section validates the integrity of all primary keys in the cleaned tables.

Each check confirms that:
- primary keys contain no missing values,
- primary keys contain no duplicate values.

These validations help ensure each table can function reliably inside a relational database.

In [ ]:
# These checks make sure each table has a unique, non-missing primary key
check_primary_key(pokemon_species, "species_id", "pokemon_species")
check_primary_key(pokemon, "pokemon_id", "pokemon")
check_primary_key(types, "type_id", "types")
check_primary_key(pokemon_types, "pokemon_type_id", "pokemon_types")
check_primary_key(moves, "move_id", "moves")
check_primary_key(pokemon_moves, "pokemon_move_id", "pokemon_moves")
check_primary_key(abilities, "ability_id", "abilities")
check_primary_key(pokemon_abilities, "pokemon_ability_id", "pokemon_abilities")
check_primary_key(stats, "stat_id", "stats")
check_primary_key(pokemon_stats, "pokemon_stat_id", "pokemon_stats")

PK check passed: pokemon_species.species_id
PK check passed: pokemon.pokemon_id
PK check passed: types.type_id
PK check passed: pokemon_types.pokemon_type_id
PK check passed: moves.move_id
PK check passed: pokemon_moves.pokemon_move_id
PK check passed: abilities.ability_id
PK check passed: pokemon_abilities.pokemon_ability_id
PK check passed: stats.stat_id
PK check passed: pokemon_stats.pokemon_stat_id


### Check Foreign Keys

This section validates relationships between tables using foreign key checks.

The workflow confirms that relationship columns correctly reference existing records in their parent tables.

These checks help detect orphan records and maintain relational database integrity across the dataset.

In [ ]:
# These checks make sure relationship columns correctly point to parent tables
check_foreign_key(pokemon, "species_id", pokemon_species, "species_id", "pokemon", "pokemon_species")
check_foreign_key(pokemon_species, "evolves_from_species_id", 
                  pokemon_species, "species_id", "pokemon_species", "pokemon_species")
check_foreign_key(pokemon_types, "pokemon_id", pokemon, "pokemon_id", "pokemon_types", "pokemon")
check_foreign_key(pokemon_types, "type_id", types, "type_id", "pokemon_types", "types")
check_foreign_key(moves, "type_id", types, "type_id", "moves", "types")
check_foreign_key(pokemon_moves, "pokemon_id", pokemon, "pokemon_id", "pokemon_moves", "pokemon")
check_foreign_key(pokemon_moves, "move_id", moves, "move_id", "pokemon_moves", "moves")
check_foreign_key(pokemon_abilities, "pokemon_id", pokemon, "pokemon_id", "pokemon_abilities", "pokemon")
check_foreign_key(pokemon_abilities, "ability_id", abilities, "ability_id", "pokemon_abilities", "abilities")
check_foreign_key(pokemon_stats, "pokemon_id", pokemon, "pokemon_id", "pokemon_stats", "pokemon")
check_foreign_key(pokemon_stats, "stat_id", stats, "stat_id", "pokemon_stats", "stats")

FK check passed: pokemon.species_id -> pokemon_species.species_id
FK check passed: pokemon_species.evolves_from_species_id -> pokemon_species.species_id
FK check passed: pokemon_types.pokemon_id -> pokemon.pokemon_id
FK check passed: pokemon_types.type_id -> types.type_id
FK check passed: moves.type_id -> types.type_id
FK check passed: pokemon_moves.pokemon_id -> pokemon.pokemon_id
FK check passed: pokemon_moves.move_id -> moves.move_id
FK check passed: pokemon_abilities.pokemon_id -> pokemon.pokemon_id
FK check passed: pokemon_abilities.ability_id -> abilities.ability_id
FK check passed: pokemon_stats.pokemon_id -> pokemon.pokemon_id
FK check passed: pokemon_stats.stat_id -> stats.stat_id


### Summarize Missing Values

This section generates a missing-value summary for all cleaned tables.

- counts missing values by column,
- prints only columns containing missing data,
- reports tables with no missing values.

This step provides a final quality check before loading the data into DuckDB.

In [ ]:
# Store missing-value counts for each cleaned table in a dictionary
missing_summary = {"pokemon_species": pokemon_species.isna().sum(),
                   "pokemon": pokemon.isna().sum(),
                   "types": types.isna().sum(),
                   "pokemon_types": pokemon_types.isna().sum(),
                   "moves": moves.isna().sum(),
                   "pokemon_moves": pokemon_moves.isna().sum(),
                   "abilities": abilities.isna().sum(),
                   "pokemon_abilities": pokemon_abilities.isna().sum(),
                   "stats": stats.isna().sum(),
                   "pokemon_stats": pokemon_stats.isna().sum()}

# Print a readable missing-value report
print("\nMissing Value Summary:")

for table_name, missing_counts in missing_summary.items():
    print(f"\n{table_name}:")

    # Only print columns that actually contain missing values
    if missing_counts.sum() > 0:
        print(missing_counts[missing_counts > 0])

    # If the table has no missing values, print a cleaner message
    else:
        print("  No missing values")


Missing Value Summary:

pokemon_species:
evolves_from_species_id    541
dtype: int64

pokemon:
base_experience    48
dtype: int64

types:
  No missing values

pokemon_types:
  No missing values

moves:
power       338
accuracy    288
pp           18
dtype: int64

pokemon_moves:
  No missing values

abilities:
  No missing values

pokemon_abilities:
  No missing values

stats:
  No missing values

pokemon_stats:
  No missing values


### Load Cleaned Tables into DuckDB

This section creates a DuckDB database and loads all cleaned pandas DataFrames into relational database tables.

- connects to the DuckDB database,
- removes old versions of tables if they exist,
- writes cleaned DataFrames directly into DuckDB.

DuckDB provides a lightweight, serverless analytical database that integrates efficiently with pandas workflows.

In [ ]:
# Open a connection to the DuckDB database
# If the database file does not exist yet, DuckDB will create it
con = duckdb.connect(duckdb_file)

# List tables in dependency-safe drop order
# Relationship tables are dropped before parent tables
tables_to_drop = ["pokemon_stats",
                  "pokemon_abilities",
                  "pokemon_moves",
                  "pokemon_types",
                  "pokemon",
                  "pokemon_species",
                  "moves",
                  "types",
                  "abilities",
                  "stats"]

# Drop existing versions of the tables so this notebook can be rerun cleanly
for tbl in tables_to_drop:
    con.execute(f"DROP TABLE IF EXISTS {tbl}")

# Create DuckDB tables directly from pandas DataFrames
con.from_df(pokemon_species).create("pokemon_species")
con.from_df(pokemon).create("pokemon")
con.from_df(types).create("types")
con.from_df(pokemon_types).create("pokemon_types")
con.from_df(moves).create("moves")
con.from_df(pokemon_moves).create("pokemon_moves")
con.from_df(abilities).create("abilities")
con.from_df(pokemon_abilities).create("pokemon_abilities")
con.from_df(stats).create("stats")
con.from_df(pokemon_stats).create("pokemon_stats")

### Confirm Tables Were Created

This section verifies that all database tables were successfully created inside DuckDB.

The `SHOW TABLES` query lists every table currently stored in the database.

This acts as a final ingestion confirmation step before running analytical SQL queries.

In [ ]:
# `SHOW TABLES` returns a list of all tables currently stored in the DuckDB database.
print("\nTables created:")
print(con.execute("SHOW TABLES").fetchall())


Tables created:
[('abilities',), ('moves',), ('pokemon',), ('pokemon_abilities',), ('pokemon_moves',), ('pokemon_species',), ('pokemon_stats',), ('pokemon_types',), ('stats',), ('types',)]


### Create SQL Views for Analysis

This section creates reusable SQL views that simplify downstream analysis.

Two views are created:
1. `pokemon_primary_type`
   - Identifies each Pokémon’s primary elemental type using the `slot = 1` relationship.

2. `pokemon_total_stats`
   - Calculates each Pokémon’s total base stats,
   - Converts individual stat rows into separate stat columns using SQL conditional aggregation.

These views make later analytical queries easier to write and interpret.

In [27]:
# Create a view that stores each Pokémon's primary type.
# The primary type is identified using slot = 1 in the pokemon_types table.
con.execute("""
CREATE OR REPLACE VIEW pokemon_primary_type AS
SELECT
    pt.pokemon_id,
    t.identifier AS primary_type
FROM pokemon_types pt
JOIN types t
    ON pt.type_id = t.type_id
WHERE pt.slot = 1
""")

# Create a view that calculates total base stats for each Pokémon
# It also pivots individual stat rows into separate stat columns
con.execute("""
CREATE OR REPLACE VIEW pokemon_total_stats AS
SELECT
    p.pokemon_id,
    p.identifier AS pokemon_name,
    p.species_id,
    SUM(ps.base_stat) AS total_base_stats,
    MAX(CASE WHEN s.identifier = 'hp' THEN ps.base_stat END) AS hp,
    MAX(CASE WHEN s.identifier = 'attack' THEN ps.base_stat END) AS attack,
    MAX(CASE WHEN s.identifier = 'defense' THEN ps.base_stat END) AS defense,
    MAX(CASE WHEN s.identifier = 'special-attack' THEN ps.base_stat END) AS special_attack,
    MAX(CASE WHEN s.identifier = 'special-defense' THEN ps.base_stat END) AS special_defense,
    MAX(CASE WHEN s.identifier = 'speed' THEN ps.base_stat END) AS speed
FROM pokemon p
JOIN pokemon_stats ps ON p.pokemon_id = ps.pokemon_id
JOIN stats s ON ps.stat_id = s.stat_id
GROUP BY p.pokemon_id, p.identifier, p.species_id;
""")

### Run Final Type-Strength Query and Export Results

This section executes the final analytical SQL query.

The analysis:
- groups Pokémon by primary type,
- calculates the average total base stats for each type,
- excludes legendary and mythical Pokémon,
- keeps only default Pokémon forms,
- filters out types with very small sample sizes.

The final results are:
- printed in the notebook,
- exported to a CSV file,
- and saved for future analysis.

The DuckDB connection is then closed to complete the workflow.

In [ ]:
# Run the final SQL analysis query and convert the output to a pandas DataFrame
results = con.execute("""
SELECT 
    ppt.primary_type AS type_name,

    -- Count how many Pokémon belong to each primary type.
    COUNT(*) AS pokemon_count,

    -- Calculate the average total base stats for each primary type.
    ROUND(AVG(pts.total_base_stats), 2) AS avg_total_base_stats

FROM pokemon_primary_type ppt

-- Join primary type information to total stat information.
JOIN pokemon_total_stats pts
    ON ppt.pokemon_id = pts.pokemon_id

-- Join back to pokemon so we can filter for default forms only.
JOIN pokemon p
    ON ppt.pokemon_id = p.pokemon_id

WHERE
    -- Keep only default Pokémon forms.
    p.is_default = TRUE

    -- Exclude legendary Pokémon species.
    AND p.species_id NOT IN (
        SELECT species_id
        FROM pokemon_species
        WHERE is_legendary = TRUE
    )

    -- Exclude mythical Pokémon species.
    AND p.species_id NOT IN (
        SELECT species_id
        FROM pokemon_species
        WHERE is_mythical = TRUE
    )

-- Summarize results by primary type.
GROUP BY ppt.primary_type

-- Only keep types with at least 5 Pokémon so averages are more stable.
HAVING COUNT(*) >= 5

-- Sort from highest average base stats to lowest.
ORDER BY avg_total_base_stats DESC;
""").df()

# Print the result table in the notebook output.
print("\nResults:")
print(results)

# Save the result table as a CSV file.
results.to_csv(os.path.join("type_strength_results.csv"), index = False)

# Close the DuckDB connection after all database work is complete.
con.close()

# Print final output messages showing where the products were saved.
print(f"\nDuckDB database saved as: {duckdb_file}")
print("Results saved as type_strength_results.csv")


Results:
   type_name  pokemon_count  avg_total_base_stats
0      steel             29                453.90
1     dragon             29                450.21
2       rock             55                433.22
3       fire             61                432.93
4       dark             38                429.03
5     ground             38                424.03
6   fighting             35                422.77
7        ice             28                421.00
8      ghost             33                419.12
9     flying              8                418.13
10  electric             52                415.04
11     fairy             26                412.54
12     water            127                409.20
13     grass             99                406.58
14    poison             37                404.08
15   psychic             42                392.69
16    normal            112                389.62
17       bug             82                371.89

DuckDB database saved as: pokemon_proje